# Grad-CAM 3D — Páncreas (`PancreasR3D18`)

Este notebook **descomprime datos** con la misma lógica que `05_Router_Vit_Lineal_Solo.ipynb` (`extract_datasets_colab`), **preprocesa volúmenes** al formato del entrenamiento (`Pancreas_R3D18_Training.ipynb`: NPZ 64³, **3 canales** + normalización **Kinetics-400**), carga el checkpoint **`r3d18_pancreas_best_V2.pth`** y aplica **Grad-CAM 3D** sobre **`layer4`**, siguiendo la técnica de `LUNA16_R3D18_Training.ipynb` (clase `GradCAM3D`, `backward` con one-hot, proyección del mapa al volumen 64³, GIF axial opcional).

**Requisitos:** `RAW_DIR` con el zip `Pancreas Cancer.zip` (o carpeta ya descomprimida en `LOCAL_DEST`) y el peso del modelo bajo `WEIGHTS_DIR`. Si no tienes `.npz`, se pueden leer **NIfTI/MHA** y generar el volumen 64³ en memoria (misma ventana HU que el EDA: [-1000, 400]). Por defecto solo se usan **2 muestras** (`NUM_SAMPLES` en la sección 3); súbelo si quieres más cortes/GIF.

## 0. Instalación

In [ ]:
!pip install -q torch torchvision numpy matplotlib scipy pillow SimpleITK tqdm

## 1. Rutas (MoE / Colab) y montaje de Drive

In [ ]:
import os
import glob
import shutil
import subprocess
import random
import io

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models.video import r3d_18, R3D_18_Weights
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from scipy.ndimage import zoom as ndzoom
from PIL import Image

try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

RAW_DIR = "/content/drive/MyDrive/PROYECTO_MOE_VISION/01_Data/Raw/"
LOCAL_DEST = "/content/datasets/"
WEIGHTS_DIR = "/content/drive/MyDrive/PROYECTO_MOE_VISION/03_Weights/"
os.makedirs(LOCAL_DEST, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

DATASET_ROOTS = {
    "Pancreas": (os.path.join(LOCAL_DEST, "Pancreas Cancer"), 4),
}

PROJ_ROOT = os.path.dirname(WEIGHTS_DIR.rstrip(os.sep))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("Pancreas root:", DATASET_ROOTS["Pancreas"][0])
print("WEIGHTS_DIR:", WEIGHTS_DIR)

## 2. Descompresión (igual que `05_Router_Vit_Lineal_Solo`)

- `only_zip_stems`: nombres del `.zip` **sin extensión**; p. ej. `("Pancreas Cancer",)` solo extrae ese dataset. `None` = todos los zip en `RAW_DIR`.
- `RUN_EXTRACT_ZIPS = False` si `LOCAL_DEST` ya tiene la carpeta descomprimida.

In [ ]:
def extract_datasets_colab(raw_dir=RAW_DIR, local_dest=LOCAL_DEST, only_zip_stems=None):
    """Copia ZIPs de Drive a LOCAL_DEST y los descomprime (misma función que 05 / GradCAM MoE)."""
    if not os.path.exists(raw_dir):
        print(f"Ruta {raw_dir} no existe. No se extrae nada.")
        return
    zip_files = sorted([f for f in os.listdir(raw_dir) if f.lower().endswith(".zip")])
    if only_zip_stems is not None:
        allow = set(only_zip_stems)
        zip_files = [f for f in zip_files if os.path.splitext(f)[0] in allow]
        print(f"Filtrado por nombre de dataset: {len(zip_files)} zip(s) -> {zip_files}")
    print(f"Encontrados {len(zip_files)} zip(s) a procesar.")
    for zip_name in zip_files:
        print("=" * 60)
        print(f"Procesando: {zip_name}")
        drive_zip_path = os.path.join(raw_dir, zip_name)
        dataset_name = os.path.splitext(zip_name)[0]
        unzip_dir = os.path.join(local_dest, dataset_name)
        local_zip_path = os.path.join(local_dest, zip_name)
        if os.path.isdir(unzip_dir) and len(os.listdir(unzip_dir)) > 0:
            print(f" Ya existe: {unzip_dir} (omitido).")
            continue
        print(" 1. Copiando ZIP...")
        shutil.copy2(drive_zip_path, local_zip_path)
        os.makedirs(unzip_dir, exist_ok=True)
        print(f" 2. Descomprimiendo en {unzip_dir}...")
        subprocess.run(["unzip", "-q", local_zip_path, "-d", unzip_dir], check=True)
        print(" 3. Borrando ZIP local.")
        os.remove(local_zip_path)
        for iz in glob.glob(os.path.join(unzip_dir, "**", "*.zip"), recursive=True):
            print(f" -> ZIP interno: {iz}")
            subprocess.run(["unzip", "-q", iz, "-d", os.path.dirname(iz)], check=True)
            os.remove(iz)
    print("\nExtracción completa.")


EXTRACT_ZIP_STEMS_ONLY = ("Pancreas Cancer",)
RUN_EXTRACT_ZIPS = True
if RUN_EXTRACT_ZIPS:
    extract_datasets_colab(only_zip_stems=EXTRACT_ZIP_STEMS_ONLY)
else:
    print("RUN_EXTRACT_ZIPS=False: se asume que LOCAL_DEST ya contiene los datasets.")

## 3. Preproceso: NPZ 64³ o NIfTI/MHA (ventana HU + resize)

Solo se **listan y preprocesan muy pocas muestras** (`NUM_SAMPLES`, por defecto **2**) para no recorrer ni cargar todo el dataset. La búsqueda de `.npz` / NIfTI se **detiene** al alcanzar ese límite.

Misma convención que `Pancreas_eda_data_preprocecing.ipynb` / `PancreasNPZDataset`: tensor **(3, 64, 64, 64)** con media/desvío **Kinetics** (como en `Pancreas_R3D18_Training.ipynb`).

In [ ]:
try:
    import SimpleITK as sitk
except ImportError:
    sitk = None

HU_MIN, HU_MAX = -1000, 400
TARGET_SIZE = (64, 64, 64)

KIN_MEAN = torch.tensor([0.43216, 0.394666, 0.37645]).view(3, 1, 1, 1)
KIN_STD = torch.tensor([0.22803, 0.22145, 0.216989]).view(3, 1, 1, 1)

# Cuántos volúmenes como máximo listar y preprocesar (Grad-CAM rápido)
NUM_SAMPLES = 2


def collect_pancreas_npz_paths(max_files):
    """Solo hasta max_files rutas .npz; deja de recorrer en cuanto alcanza el límite."""
    roots = [
        DATASET_ROOTS["Pancreas"][0],
        "/content/dataset_npz",
        os.path.join(PROJ_ROOT, "dataset_npz"),
    ]
    seen = set()
    out = []
    for r in roots:
        if not r or not os.path.isdir(r) or len(out) >= max_files:
            continue
        for dirpath, _, filenames in os.walk(r):
            if len(out) >= max_files:
                break
            for fn in sorted(filenames):
                if len(out) >= max_files:
                    break
                if fn.lower().endswith(".npz"):
                    p = os.path.join(dirpath, fn)
                    if p not in seen:
                        seen.add(p)
                        out.append(p)
    return out


def scan_pancreas_raw_volume_paths(root_dir, max_files):
    """Solo hasta max_files NIfTI/MHA; parada anticipada."""
    out = []
    if not root_dir or not os.path.isdir(root_dir):
        return out
    for dirpath, _, filenames in os.walk(root_dir):
        if len(out) >= max_files:
            break
        for fn in sorted(filenames):
            if len(out) >= max_files:
                break
            low = fn.lower()
            if low.endswith(".nii.gz") or low.endswith(".nii") or low.endswith(".mha"):
                out.append(os.path.join(dirpath, fn))
    return out


def preprocess_volume_from_nifti_mha(file_path):
    if sitk is None:
        raise RuntimeError("Instala SimpleITK: pip install SimpleITK")
    img = sitk.ReadImage(file_path)
    arr_raw = sitk.GetArrayFromImage(img).astype(np.float32)
    del img
    np.clip(arr_raw, HU_MIN, HU_MAX, out=arr_raw)
    arr_raw -= HU_MIN
    arr_raw /= (HU_MAX - HU_MIN)
    with torch.no_grad():
        t = torch.from_numpy(arr_raw).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(t, size=TARGET_SIZE, mode="trilinear", align_corners=False)
        vol = t.squeeze().numpy().copy()
    return vol.astype(np.float32)


def volume_to_kinetics_tensor(vol_64):
    """vol: (64,64,64) float [0,1] -> tensor (3,64,64,64) normalizado Kinetics."""
    v = torch.from_numpy(vol_64.astype(np.float32))
    v3 = v.unsqueeze(0).repeat(3, 1, 1, 1)
    v3 = (v3 - KIN_MEAN) / KIN_STD
    return v3


def load_volume_and_label_npz(path):
    d = np.load(path, allow_pickle=True)
    vol = d["volume"].astype(np.float32)
    if "label" in d:
        y = int(np.asarray(d["label"]).ravel()[0])
    elif "y" in d:
        y = int(np.asarray(d["y"]).ravel()[0])
    else:
        y = 0
    return vol, y


def infer_label_from_path(path):
    low = path.lower()
    for t in ["cancer", "tumor", "positive", "pdac", "malig"]:
        if t in low:
            return 1
    for t in ["normal", "negative", "benign", "non_tumor"]:
        if t in low:
            return 0
    return 0


npz_list = collect_pancreas_npz_paths(NUM_SAMPLES)
print(f"NPZ listados (máx. {NUM_SAMPLES}):", len(npz_list))

random.seed(42)

if len(npz_list) >= 1:
    use_npz = True
    pool = npz_list
    print("Modo: muestras desde archivos .npz")
else:
    use_npz = False
    pool = scan_pancreas_raw_volume_paths(DATASET_ROOTS["Pancreas"][0], NUM_SAMPLES)
    print("Modo: sin NPZ; volúmenes crudos listados (máx. %d):" % NUM_SAMPLES, len(pool))
    if len(pool) < 1:
        raise RuntimeError(
            "No hay .npz ni NIfTI/MHA bajo la carpeta de páncreas. "
            "Genera NPZ con el EDA o descomprime el zip del dataset en LOCAL_DEST."
        )

n_pick = min(NUM_SAMPLES, len(pool))
sample_paths = random.sample(pool, n_pick)
print(f"Preprocesamiento Grad-CAM solo sobre {len(sample_paths)} volumen(es):", sample_paths)

## 4. Modelo `PancreasR3D18` (idéntico al notebook de entrenamiento)

In [ ]:
NUM_CLASSES = 2


class PancreasR3D18(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        weights = R3D_18_Weights.DEFAULT if pretrained else None
        backbone = r3d_18(weights=weights)
        self.stem = backbone.stem
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.avgpool = backbone.avgpool
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )
        self.use_checkpointing = True

    def forward(self, x):
        x = self.stem(x)
        if self.use_checkpointing and self.training:
            x = grad_checkpoint(self._run_layer1, x, use_reentrant=False)
            x = grad_checkpoint(self._run_layer2, x, use_reentrant=False)
            x = grad_checkpoint(self._run_layer3, x, use_reentrant=False)
            x = grad_checkpoint(self._run_layer4, x, use_reentrant=False)
        else:
            x = self.layer1(x)
            x = self.layer2(x)
            x = self.layer3(x)
            x = self.layer4(x)
        x = self.avgpool(x)
        x = x.flatten(1)
        return self.head(x)

    def _run_layer1(self, x):
        return self.layer1(x)

    def _run_layer2(self, x):
        return self.layer2(x)
    def _run_layer3(self, x):
        return self.layer3(x)
    def _run_layer4(self, x):
        return self.layer4(x)


def resolve_pancreas_training_checkpoint(base_dir):
    """Rutas habituales: 03_Weights (MoE) o Weights (notebook de entrenamiento)."""
    name = "r3d18_pancreas_best_V2.pth"
    cands = [
        os.path.join(base_dir, name),
        os.path.join(PROJ_ROOT, "03_Weights", name),
        os.path.join(PROJ_ROOT, "Weights", name),
        os.path.join(PROJ_ROOT, "03_Weights", "Experts_3D", name),
    ]
    for c in cands:
        if c and os.path.isfile(c):
            return c
    return None


model = PancreasR3D18(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
ckpt_path = resolve_pancreas_training_checkpoint(WEIGHTS_DIR)
if not ckpt_path:
    raise FileNotFoundError(
        f"No se encontró r3d18_pancreas_best_V2.pth bajo {WEIGHTS_DIR} ni PROJ_ROOT."
    )
try:
    sd = torch.load(ckpt_path, map_location="cpu", weights_only=True)
except TypeError:
    sd = torch.load(ckpt_path, map_location="cpu")
model.load_state_dict(sd, strict=True)
model.eval()
print("Checkpoint:", ckpt_path)

## 5. Grad-CAM 3D (como `LUNA16_R3D18_Training.ipynb`)

Capa objetivo: **`model.layer4`**. Mapa reescalado a 64³ con `scipy.ndimage.zoom`, visualización axial y GIF opcional.

In [ ]:
try:
    __import__("IPython").get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass


class GradCAM3D:
    """Grad-CAM genérico para tensores (B,C,D,H,W) [R3D backbone] — LUNA16 notebook."""

    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._fwd_hook)
        target_layer.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, module, inp, out):
        self.activations = out.detach()

    def _bwd_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def __call__(self, x, class_idx=None):
        self.model.eval()
        x = x.clone().requires_grad_(True)
        logits = self.model(x)
        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()
        self.model.zero_grad()
        one_hot = torch.zeros_like(logits)
        one_hot[0, class_idx] = 1.0
        logits.backward(gradient=one_hot)

        act = self.activations
        grad = self.gradients
        if act is None or grad is None:
            raise RuntimeError("Grad-CAM: activaciones o gradientes None.")

        if act.ndim == 4:
            weights = grad.mean(dim=[1, 2], keepdim=True)
            cam = (weights * act).sum(dim=-1)
        elif act.ndim == 5:
            weights = grad.mean(dim=[2, 3, 4], keepdim=True)
            cam = (weights * act).sum(dim=1)
        else:
            weights = grad.mean(dim=1, keepdim=True)
            cam = (weights * act).sum(dim=-1)

        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam[0].cpu().numpy(), class_idx


def cam_to_volume_64(cam_raw):
    """Interpola el mapa Grad-CAM al volumen 64³ (LUNA16)."""
    if cam_raw.ndim == 2:
        cam_vol = ndzoom(cam_raw, np.array([64, 64]) / np.array(cam_raw.shape), order=1)
        cam_vol = np.stack([cam_vol] * 64, axis=0)
    elif cam_raw.ndim == 3:
        factors = np.array([64, 64, 64]) / np.array(cam_raw.shape)
        cam_vol = ndzoom(cam_raw, factors, order=1)
    else:
        cam_vol = np.zeros((64, 64, 64), dtype=np.float32)
    return cam_vol.astype(np.float32)


def save_axial_gif(vol_np, cam_vol, out_path, title="", fps=10):
    D = vol_np.shape[0]
    vol_min, vol_max = float(vol_np.min()), float(vol_np.max())
    duration_ms = max(40, int(1000 / max(1, fps)))
    frames = []
    for z in range(D):
        fig, ax = plt.subplots(figsize=(5.5, 5.5), dpi=110)
        ax.imshow(vol_np[z], cmap="gray", vmin=vol_min, vmax=vol_max)
        ax.imshow(cam_vol[z], cmap="jet", alpha=0.42, vmin=0.0, vmax=1.0)
        ax.set_title(f"{title}  axial  z={z:02d}/{D - 1}")
        ax.axis("off")
        buf = io.BytesIO()
        fig.savefig(buf, format="png", bbox_inches="tight", pad_inches=0.08, facecolor="white")
        plt.close(fig)
        buf.seek(0)
        frames.append(Image.open(buf).convert("RGB"))
    frames[0].save(
        out_path,
        save_all=True,
        append_images=frames[1:],
        duration=duration_ms,
        loop=0,
    )


target_layer = model.layer4
gradcam = GradCAM3D(model, target_layer)

fig_dir = os.path.join(WEIGHTS_DIR, "figures", "pancreas_r3d18_gradcam")
os.makedirs(fig_dir, exist_ok=True)

name_cls = {0: "clase0", 1: "clase1_PDAC_o_pos"}

for si, pth in enumerate(sample_paths):
    if use_npz:
        vol_raw, y_true = load_volume_and_label_npz(pth)
    else:
        vol_raw = preprocess_volume_from_nifti_mha(pth)
        y_true = infer_label_from_path(pth)

    x_chw = volume_to_kinetics_tensor(vol_raw).unsqueeze(0).to(DEVICE)

    cam_raw, cls_used = gradcam(x_chw, class_idx=None)
    cam_vol = cam_to_volume_64(cam_raw)

    with torch.no_grad():
        pred = int(model(x_chw).argmax(dim=1).item())

    vol_vis = np.asarray(vol_raw, dtype=np.float32)
    vmn, vmx = float(vol_vis.min()), float(vol_vis.max())
    vol_vis = (vol_vis - vmn) / (vmx - vmn + 1e-8)

    base = os.path.splitext(os.path.basename(pth))[0][:40]
    gif_path = os.path.join(fig_dir, f"gradcam_pancreas_{si:02d}_{name_cls.get(y_true, 'y')}_{base}.gif")
    title_gif = f"GT={y_true} Pred={pred} target_cls={cls_used}"
    save_axial_gif(vol_vis, cam_vol, gif_path, title=title_gif, fps=8)
    print("GIF:", gif_path)

    mid = [s // 2 for s in vol_vis.shape]
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
    slices_info = [
        ("Axial", vol_vis[mid[0]], cam_vol[mid[0]]),
        ("Coronal", vol_vis[:, mid[1]], cam_vol[:, mid[1]]),
        ("Sagital", vol_vis[:, :, mid[2]], cam_vol[:, :, mid[2]]),
    ]
    for j, (name, sl, cam_sl) in enumerate(slices_info):
        axes[j].imshow(sl, cmap="gray")
        axes[j].imshow(cam_sl, cmap="jet", alpha=0.4)
        axes[j].set_title(f"{name} | {base} | GT={y_true} Pred={pred}")
        axes[j].axis("off")
    plt.suptitle("PancreasR3D18 — Grad-CAM 3D (layer4)", fontsize=12)
    plt.tight_layout()
    plt.show()

## Nota

- La celda de carga usa `weights_only=True` si está disponible; si no, recae en `torch.load` clásico.
- La clase objetivo del Grad-CAM por defecto es **argmax** del logit; puedes fijar `class_idx=0` o `1` en `gradcam(x, class_idx=...)` para forzar la explicación respecto a esa clase.